## Setup and Configuration

In [16]:
import sys
import subprocess
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Add workspace root to path
workspace_root = Path().resolve().parent.parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print(f"Workspace root: {workspace_root}")

Workspace root: /Users/lukas/Workspace/Uni/BachelorsThesis/FICE


## Configuration Parameters

Modify these parameters for your dataset:

In [17]:
# Dataset configuration
DATASET_NAME = "swdf_star"
GRAPH_PATH = workspace_root / "data/swdf/raw/59320634"
QUERY_PATH = workspace_root / "data/swdf_star_queries/raw/59320652"
QUERY_TYPE = "star"  # For results organization

# RDF type URI (auto-detected if None)
RDF_TYPE_URI = None  # Will use swdf default: '<http://ex.org/03>'

# Paths
neuso_benchmark_dir = Path(".").resolve()
neuso_root = neuso_benchmark_dir / "NeuSO"
results_dir = neuso_benchmark_dir / "results" / QUERY_TYPE / DATASET_NAME / "neuso"

print(f"Dataset: {DATASET_NAME}")
print(f"Graph: {GRAPH_PATH}")
print(f"Queries: {QUERY_PATH}")
print(f"Results will be saved to: {results_dir}")

Dataset: swdf_star
Graph: /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf/raw/59320634
Queries: /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf_star_queries/raw/59320652
Results will be saved to: /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/neuso/results/star/swdf_star/neuso


## Step 1: Data Preparation

Transform TTL graph and query JSON to NeuSO format.

In [18]:
# Build command
# TODO: change
cmd = [
    sys.executable,
    "prepare_data.py",
    "--dataset-name", DATASET_NAME,
    "--graph-path", str(GRAPH_PATH),
    "--query-path", str(QUERY_PATH),
]

if RDF_TYPE_URI:
    cmd.extend(["--rdf-type-uri", RDF_TYPE_URI])

print("Running data preparation...")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, cwd=neuso_benchmark_dir)

if result.returncode == 0:
    print("\n✓ Data preparation completed successfully!")
else:
    print("\n✗ Data preparation failed!")
    raise RuntimeError("Data preparation failed")

Running data preparation...
/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/.venv/bin/python prepare_data.py --dataset-name swdf_star --graph-path /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf/raw/59320634 --query-path /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf_star_queries/raw/59320652

Using rdf:type URI: <http://ex.org/03>

Preparing NeuSO dataset: swdf_star
Graph: /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf/raw/59320634
Queries: /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf_star_queries/raw/59320652
Output: /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/neuso/NeuSO/dataset/swdf_star


--- Step 1: Transforming Data Graph ---
Reading graph from /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf/raw/59320634...
Pass 1: Extracting vertices and predicates...
Creating vertex mappings...
Creating predicate mappings...
Pass 2: Processing vertex types...
Pass 3: Creating edge list and collecting stats...


100%|██████████| 242256/242256 [00:00<00:00, 1770049.79it/s]


Writing NeuSO graph to /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/neuso/NeuSO/dataset/swdf_star/data_graph/swdf_star.graph...
Saving mappings and stats...
Generating random label embeddings...
Saved label embeddings to /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/neuso/NeuSO/dataset/swdf_star/data_graph/swdf_star.emb.npy
Embedding shape: (119, 128)
Transformation complete!
✓ Graph transformation complete
  - Vertices: 76711
  - Edges: 219735
  - Predicates: 170

--- Step 2: Transforming Queries ---
Loading mappings...
Loading queries from /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/data/swdf_star_queries/raw/59320652...
Processing 116645 queries...
Query transformation complete.
✓ Query transformation complete

--- Step 3: Updating NeuSO Config ---
Updated config.yaml: dataset='swdf_star', topology_emb=True
✓ Config update complete

--- Step 4: Verifying Generated Files ---
  ✓ swdf_star.graph
  ✓ swdf_star.emb.npy
  ✓ id_to_id_mapping.json
  ✓ id_to_i

### Verify Generated Files

In [19]:
dataset_dir = neuso_root / "dataset" / DATASET_NAME

print("Generated files:")
print(f"\nData graph:")
for f in (dataset_dir / "data_graph").glob("*"):
    print(f"  {f.name}")

print(f"\nQuery files:")
query_files = list((dataset_dir / "query_graph").glob("*.graph"))
print(f"  {len(query_files)} query graph files")

filter_files = list((dataset_dir / "filter").glob("*.filter"))
print(f"  {len(filter_files)} filter files")

ccg_files = list((dataset_dir / "ccg").glob("*.ccg"))
print(f"  {len(ccg_files)} ccg files")

print(f"\nMappings:")
for f in dataset_dir.glob("*.json"):
    print(f"  {f.name}")

Generated files:

Data graph:
  swdf_star.emb.npy
  swdf_star.graph

Query files:
  116645 query graph files
  116645 filter files
  116645 ccg files

Mappings:
  id_to_id_mapping_predicate.json
  label_to_id_mapping.json
  id_to_id_mapping.json
  stats.json


## Step 2: Model Training

Train NeuSO model on the prepared dataset.

**Note:** This can take a significant amount of time (hours) depending on the dataset size and number of epochs (default: 100).

In [20]:
cmd = [
    sys.executable,
    "train_neuso.py",
    "--dataset", DATASET_NAME,
    "--mode", "train"
]

print("Starting model training...")
print(" ".join(cmd))
print("\nThis may take a while. Training progress will be displayed below.\n")
print("="*60)

result = subprocess.run(cmd, cwd=neuso_benchmark_dir)

if result.returncode == 0:
    print("\n✓ Training completed successfully!")
else:
    print("\n✗ Training failed!")
    raise RuntimeError("Training failed")

Starting model training...
/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/.venv/bin/python train_neuso.py --dataset swdf_star --mode train

This may take a while. Training progress will be displayed below.


✗ Training failed!


Traceback (most recent call last):
  File "/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/neuso/train_neuso.py", line 13, in <module>
    from NeuSO.parser_config import ConfigParser
  File "/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/neuso/NeuSO/parser_config.py", line 3, in <module>
    from Logger import setup_logging
ModuleNotFoundError: No module named 'Logger'


RuntimeError: Training failed

## Step 3: Inference and Evaluation

Run cardinality estimation on test queries and save results.

In [ ]:
cmd = [
    sys.executable,
    "run_estimation.py",
    "--dataset", DATASET_NAME,
    "--query-type", QUERY_TYPE,
]

print("Running inference...")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, cwd=neuso_benchmark_dir)

if result.returncode == 0:
    print("\n✓ Inference completed successfully!")
else:
    print("\n✗ Inference failed!")
    raise RuntimeError("Inference failed")

Running inference...
/Users/lukas/Workspace/Uni/BachelorsThesis/FICE/.venv/bin/python run_estimation.py --dataset swdf_star --query-type star

Error: No model checkpoints found in /Users/lukas/Workspace/Uni/BachelorsThesis/FICE/benchmarks/neuso/NeuSO/saved
Please train a model first using train_neuso.py

✗ Inference failed!


RuntimeError: Inference failed

## Step 4: Load and Analyze Results

In [ ]:
# Load results
gts = np.load(results_dir / "gts.npy")
preds = np.load(results_dir / "preds.npy")
pred_times = np.load(results_dir / "pred_times.npy")
sizes = np.load(results_dir / "sizes.npy")

with open(results_dir / "results.json", "r") as f:
    results = json.load(f)

print("Results Summary:")
print("="*60)
print(f"Method: {results['method']}")
print(f"Number of queries: {results['num_queries']}")
print(f"Median Q-Error: {results['median_q_error']:.4f}")
print(f"Mean Q-Error: {results['mean_q_error']:.4f}")
print(f"MSE: {results['mse']:.4f}")
print(f"MAE: {results['mae']:.4f}")
print(f"Mean prediction time: {results['mean_prediction_time_seconds']:.6f}s")
print(f"Total prediction time: {results['total_prediction_time_seconds']:.4f}s")
print("="*60)

## Step 5: Visualization

In [ ]:
# Calculate Q-Error
q_errors = np.maximum(preds / (gts + 1e-10), gts / (preds + 1e-10))

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Predictions vs Ground Truth
ax = axes[0, 0]
ax.scatter(gts, preds, alpha=0.5)
ax.plot([gts.min(), gts.max()], [gts.min(), gts.max()], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Ground Truth Cardinality')
ax.set_ylabel('Predicted Cardinality')
ax.set_title('Predictions vs Ground Truth')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 2. Q-Error Distribution
ax = axes[0, 1]
ax.hist(q_errors, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(np.median(q_errors), color='r', linestyle='--', linewidth=2, label=f'Median: {np.median(q_errors):.2f}')
ax.set_xlabel('Q-Error')
ax.set_ylabel('Frequency')
ax.set_title('Q-Error Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Q-Error vs Query Size
ax = axes[1, 0]
ax.scatter(sizes, q_errors, alpha=0.5)
ax.set_xlabel('Query Size (number of triples)')
ax.set_ylabel('Q-Error')
ax.set_title('Q-Error vs Query Size')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 4. Prediction Time Distribution
ax = axes[1, 1]
ax.hist(pred_times * 1000, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(np.mean(pred_times) * 1000, color='r', linestyle='--', linewidth=2, 
           label=f'Mean: {np.mean(pred_times)*1000:.2f}ms')
ax.set_xlabel('Prediction Time (ms)')
ax.set_ylabel('Frequency')
ax.set_title('Prediction Time Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / "neuso_results.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Visualization saved to: {results_dir / 'neuso_results.png'}")

## Step 6: Compare with Other Methods (Optional)

If you have results from other methods (e.g., gcare), load and compare them here.

In [ ]:
# Load gcare results for comparison
gcare_results_dir = workspace_root / "benchmarks/gcare/results" / QUERY_TYPE / DATASET_NAME

methods_to_compare = []

# Check which gcare methods have results
if gcare_results_dir.exists():
    for method_dir in gcare_results_dir.iterdir():
        if method_dir.is_dir() and (method_dir / "results.json").exists():
            with open(method_dir / "results.json", "r") as f:
                method_results = json.load(f)
                methods_to_compare.append({
                    'name': method_dir.name,
                    'median_q_error': method_results['median_q_error'],
                    'mean_time': method_results['mean_prediction_time_seconds']
                })

# Add NeuSO results
methods_to_compare.append({
    'name': 'neuso',
    'median_q_error': results['median_q_error'],
    'mean_time': results['mean_prediction_time_seconds']
})

if len(methods_to_compare) > 1:
    # Create comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    names = [m['name'] for m in methods_to_compare]
    q_errors_comp = [m['median_q_error'] for m in methods_to_compare]
    times_comp = [m['mean_time'] * 1000 for m in methods_to_compare]  # Convert to ms
    
    # Median Q-Error comparison
    ax = axes[0]
    bars = ax.bar(names, q_errors_comp)
    bars[-1].set_color('orange')  # Highlight NeuSO
    ax.set_ylabel('Median Q-Error')
    ax.set_title('Median Q-Error Comparison')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, axis='y')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Prediction time comparison
    ax = axes[1]
    bars = ax.bar(names, times_comp)
    bars[-1].set_color('orange')  # Highlight NeuSO
    ax.set_ylabel('Mean Prediction Time (ms)')
    ax.set_title('Prediction Time Comparison')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, axis='y')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig(results_dir / "method_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nComparison saved to: {results_dir / 'method_comparison.png'}")
else:
    print("No other method results found for comparison.")

## Summary

The complete NeuSO pipeline has been executed:
1. ✓ Data preparation
2. ✓ Model training
3. ✓ Inference and evaluation
4. ✓ Results visualization

All results are saved in the `results/` directory in the same format as gcare benchmarks.